In [2]:
from sentence_transformers import SentenceTransformer
import torch
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT loaded")

print("Torch version:", torch.__version__)
print("FAISS version:", faiss.__version__)

c:\Users\pande\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\pande\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pande\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

SBERT loaded
Torch version: 2.9.1+cpu
FAISS version: 1.13.2


In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

from sentence_transformers import SentenceTransformer
import faiss

In [5]:
df = pd.read_csv(
    "./Combined_Jobs_Final.csv",
    engine="python",
    on_bad_lines="skip"
)

In [6]:
raw_df = pd.read_csv(
    "./Combined_Jobs_Final.csv",
    engine="python",
    on_bad_lines="skip"
)

print("Rows loaded:", raw_df.shape[0])
print("Columns loaded:", raw_df.shape[1])

Rows loaded: 84090
Columns loaded: 23


In [7]:
print(df.shape)
df.head()

(84090, 23)


,Job.ID,Provider,Status,Slug,Title,Position,Company,City,State.Name,State.Code,...,Industry,Job.Description,Requirements,Salary,Listing.Start,Listing.End,Employment.Type,Education.Required,Created.At,Updated.At
0,111,1,open,palo-alto-ca-tacolicious-server,Server @ Tacolicious,Server,Tacolicious,Palo Alto,California,CA,...,Food and Beverages,Tacolicious' first Palo Alto store just opened...,NaN,8.00,NaN,NaN,Part-Time,NaN,2013-03-12 02:08:28 UTC,2014-08-16 15:35:36 UTC
1,113,1,open,san-francisco-ca-claude-lane-kitchen-staff-chef,Kitchen Staff/Chef @ Claude Lane,Kitchen Staff/Chef,Claude Lane,San Francisco,California,CA,...,Food and Beverages,\r\n\r\nNew French Brasserie in S.F. Financia...,NaN,0.00,NaN,NaN,Part-Time,NaN,2013-04-12 08:36:36 UTC,2014-08-16 15:35:36 UTC
2,117,1,open,san-francisco-ca-machka-restaurants-corp-barte...,Bartender @ Machka Restaurants Corp.,Bartender,Machka Restaurants Corp.,San Francisco,California,CA,...,Food and Beverages,We are a popular Mediterranean wine bar and re...,NaN,11.00,NaN,NaN,Part-Time,NaN,2013-07-16 09:34:10 UTC,2014-08-16 15:35:37 UTC
3,121,1,open,brisbane-ca-teriyaki-house-server,Server @ Teriyaki House,Server,Teriyaki House,Brisbane,California,CA,...,Food and Beverages,● Serve food/drinks to customers in a profess...,NaN,10.55,NaN,NaN,Part-Time,NaN,2013-09-04 15:40:30 UTC,2014-08-16 15:35:38 UTC
4,127,1,open,los-angeles-ca-rosa-mexicano-sunset-kitchen-st...,Kitchen Staff/Chef @ Rosa Mexicano - Sunset,Kitchen Staff/Chef,Rosa Mexicano - Sunset,Los Angeles,California,CA,...,Food and Beverages,"Located at the heart of Hollywood, we are one ...",NaN,10.55,NaN,NaN,Part-Time,NaN,2013-07-17 15:26:18 UTC,2014-08-16 15:35:40 UTC


In [8]:
df = df[
    [
        "Title",
        "Company",
        "City",
        "Position",
        "Industry",
        "Employment.Type",
        "Job.Description",
        "Requirements"
    ]
]

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [10]:
df["combined_text"] = (
    df["Title"].fillna("") + " " +
    df["Position"].fillna("") + " " +
    df["Job.Description"].fillna("") + " " +
    df["Requirements"].fillna("")
)

df["combined_text"] = df["combined_text"].apply(clean_text)

In [11]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(df["combined_text"])

In [12]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = sbert_model.encode(
    df["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

Batches: 100%|██████████| 1314/1314 [1:00:45<00:00,  2.77s/it]


In [13]:
title_embeddings = sbert_model.encode(
    df["Title"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True
)

title_embeddings = title_embeddings.astype("float32")
faiss.normalize_L2(title_embeddings)

title_index = faiss.IndexFlatIP(title_embeddings.shape[1])
title_index.add(title_embeddings)

Batches: 100%|██████████| 1314/1314 [05:07<00:00,  4.27it/s]


In [14]:
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

In [15]:
SKILLS = [
    "customer service", "communication", "teamwork",
    "time management", "problem solving",
    "server", "bartender", "chef", "kitchen",
    "food preparation", "cleaning",
    "supervisor", "manager", "scheduling",
    "excel", "ms office", "computer skills"
]

In [16]:
def skill_score(query_text, candidate_text):
    s1 = set(skill for skill in SKILLS if skill in query_text)
    s2 = set(skill for skill in SKILLS if skill in candidate_text)

    if not s1:
        return 0.0
    return len(s1 & s2) / len(s1)

In [18]:
def recommend_jobs(title_query, top_k=10, ann_k=50):
    # --- Step 0: Find closest title semantically ---
    qemb = sbert_model.encode([title_query]).astype("float32")
    faiss.normalize_L2(qemb)

    _, title_idx = title_index.search(qemb, 1)
    qidx = title_idx[0][0]

    matched_title = df.iloc[qidx]["Title"]

    # --- Step 1: SBERT ANN search ---
    qvec = embeddings[qidx].reshape(1, -1)
    scores, indices = index.search(qvec, ann_k + 1)

    candidate_idx = indices[0][1:]
    sbert_scores = scores[0][1:]

    # --- Step 2: TF-IDF scoring ---
    tfidf_scores = cosine_similarity(
        tfidf_matrix[qidx],
        tfidf_matrix[candidate_idx]
    ).flatten()

    # --- Step 3: Skill scoring ---
    qtext = df.iloc[qidx]["combined_text"]
    skill_scores = np.array([
        skill_score(qtext, df.iloc[i]["combined_text"])
        for i in candidate_idx
    ])

    # --- Step 4: Normalize ---
    # --- Step 4: Normalize ---
    def norm(x):
      return (x - x.min()) / (np.ptp(x) + 1e-6)

    sbert_scores = norm(sbert_scores)
    tfidf_scores = norm(tfidf_scores)


    # --- Step 5: Final hybrid score ---
    final_scores = (
        0.5 * sbert_scores +
        0.3 * tfidf_scores +
        0.2 * skill_scores
    )

    top = np.argsort(final_scores)[-top_k:][::-1]

    results = df.iloc[candidate_idx[top]][[
        "Title", "Company", "City", "Position", "Industry", "Employment.Type"
    ]]

    print(f"Query matched to dataset title: '{matched_title}'")

    return results

In [19]:
import numpy as np

def precision_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / len(relevant) if relevant else 0

def ndcg_at_k(recommended, relevant, k):
    score = 0.0
    for i, job in enumerate(recommended[:k]):
        if job in relevant:
            score += 1 / np.log2(i + 2)
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return score / ideal if ideal > 0 else 0

In [20]:
def get_relevant_jobs(idx, df):
    industry = df.iloc[idx]["Industry"]
    position = df.iloc[idx]["Position"]

    relevant = df[
        (df["Industry"] == industry) |
        (df["Position"] == position)
    ].index.tolist()

    relevant.remove(idx)
    return relevant

In [21]:
import random

def evaluate_model(df, recommend_fn, sample_size=500, k=10):
    indices = random.sample(range(len(df)), sample_size)

    precisions, recalls, ndcgs = [], [], []

    for idx in indices:
        results = recommend_fn(df.iloc[idx]["Title"], top_k=k)
        recommended = results.index.tolist()
        relevant = get_relevant_jobs(idx, df)

        precisions.append(precision_at_k(recommended, relevant, k))
        recalls.append(recall_at_k(recommended, relevant, k))
        ndcgs.append(ndcg_at_k(recommended, relevant, k))

    return {
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "NDCG@K": np.mean(ndcgs)
    }

In [22]:
metrics = evaluate_model(df, recommend_jobs, sample_size=500, k=10)
metrics

Query matched to dataset title: 'Administrative Coordinator @ Holy Cross Health'
Query matched to dataset title: 'Data Entry Clerk / Data Entry Specialist (Data Entry) @ OfficeTeam'
Query matched to dataset title: 'Area Sales Representative @ CHI Payment Systems'
Query matched to dataset title: 'COSMETOLOGY / SALON CAREER TRAINING - LOCAL HAIR / MAKEUP TRAINING AVAILABLE @ My Cosmetology Career'
Query matched to dataset title: 'Radiology Technologist @ Encompass Medical Group'
Query matched to dataset title: 'Registered Nurse – Critical Care @ Swedish Health'
Query matched to dataset title: 'Part-time Automotive Service Writer @ Rod Baker Ford'
Query matched to dataset title: 'PHARMACY TECHNICIAN @ Fred Meyer'
Query matched to dataset title: 'Part time Administrative Assistant @ Lyneer Staffing Solutions'
Query matched to dataset title: 'Maintenance Worker @ Service Corporation International'
Query matched to dataset title: 'Manager of Valet Services -  Driskill Hotel @ Towne Park'
Que

{'Precision@K': np.float64(0.37360000000000004),
 'Recall@K': np.float64(0.16144568742020132),
 'NDCG@K': np.float64(0.4367035501571238)}

In [23]:
def recommend_tfidf(title_query, top_k=10):
    # Find first matching title using plain-text match (NOT regex)
    title_idx = df[
        df["Title"].str.contains(
            title_query,
            case=False,
            na=False,
            regex=False
        )
    ].index

    if len(title_idx) == 0:
        qidx = 0  # safe fallback
    else:
        qidx = title_idx[0]

    sims = cosine_similarity(
        tfidf_matrix[qidx],
        tfidf_matrix
    ).flatten()

    sims[qidx] = -1  # remove self
    top_idx = np.argsort(sims)[-top_k:][::-1]

    return df.iloc[top_idx]

In [24]:
def recommend_sbert_only(title_query, top_k=10):
    qemb = sbert_model.encode([title_query]).astype("float32")
    faiss.normalize_L2(qemb)

    scores, indices = index.search(qemb, top_k + 1)
    top_idx = indices[0][1:]  # remove self

    return df.iloc[top_idx]

In [25]:
tfidf_metrics = evaluate_model(
    df,
    recommend_tfidf,
    sample_size=500,
    k=10
)

sbert_metrics = evaluate_model(
    df,
    recommend_sbert_only,
    sample_size=500,
    k=10
)

hybrid_metrics = evaluate_model(
    df,
    recommend_jobs,
    sample_size=500,
    k=10
)

tfidf_metrics, sbert_metrics, hybrid_metrics

Query matched to dataset title: 'Computer Operator'
Query matched to dataset title: 'Sales Representative / Sales Associate ( Entry Level ) @ Vector Marketing'
Query matched to dataset title: 'ASSISTANT MANAGER @ Dollar Tree Stores'
Query matched to dataset title: 'Transaction Support Associate II @ Fifth Third Bank'
Query matched to dataset title: 'Food Service Worker I @ Sodexo at Kennesaw State University'
Query matched to dataset title: 'Sales Marketing Retail Representative(Retail Marketing) @ Bluegreen Vacations'
Query matched to dataset title: 'Guest Service Agent - Auburn NY @ Holiday Inn- Independently Owned & Operated'
Query matched to dataset title: 'Pharmacy Tech II @ The US Oncology Network'
Query matched to dataset title: 'Accounting Clerk @ Accountemps'
Query matched to dataset title: 'Registered Nurse (RN) / Licensed Practical Nurse (LPN) - Healthcare Nursing Staff'
Query matched to dataset title: 'Data Entry Position- Kelly Services @ Kelly Services'
Query matched to d

({'Precision@K': np.float64(0.3632),
  'Recall@K': np.float64(0.15548667710489866),
  'NDCG@K': np.float64(0.42816895877062144)},
 {'Precision@K': np.float64(0.332),
  'Recall@K': np.float64(0.14747493564337882),
  'NDCG@K': np.float64(0.37572664666891786)},
 {'Precision@K': np.float64(0.3994),
  'Recall@K': np.float64(0.16737401684364403),
  'NDCG@K': np.float64(0.4675896767899601)})

In [26]:
recommend_jobs("Customer Service Associate", top_k=10)

Query matched to dataset title: 'Customer Service Representative/Sales Associate'


,Title,Company,City,Position,Industry,Employment.Type
14272,Customer Service Representative/Sales Associate,NaN,Venice,Customer Service Representative/Sales Associate,NaN,Part-Time
17016,Customer Service Representative/Sales Associat...,Speedway LLC,Newton,Customer Service Representative/Sales Associate,NaN,Part-Time
15708,Customer Service Representative/Sales Associat...,Speedway LLC,Dedham,Customer Service Representative/Sales Associate,NaN,Part-Time
15717,Customer Service Representative/Sales Associat...,Speedway LLC,Woburn,Customer Service Representative/Sales Associate,NaN,Part-Time
15719,Customer Service Representative/Sales Associat...,Speedway LLC,Framingham,Customer Service Representative/Sales Associate,NaN,Part-Time
15718,Customer Service Representative/Sales Associat...,Speedway LLC,Winchester,Customer Service Representative/Sales Associate,NaN,Part-Time
15721,Customer Service Representative/Sales Associat...,Speedway LLC,Hampton,Customer Service Representative/Sales Associate,NaN,Part-Time
15722,Customer Service Representative/Sales Associat...,Speedway LLC,Worcester,Customer Service Representative/Sales Associate,NaN,Part-Time
15716,Customer Service Representative/Sales Associat...,Speedway LLC,Natick,Customer Service Representative/Sales Associate,NaN,Part-Time
15715,Customer Service Representative/Sales Associat...,Speedway LLC,New Bedford,Customer Service Representative/Sales Associate,NaN,Part-Time
